In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *

from datetime import datetime, date
dbutils.widgets.text('season',"2024","ipl season")
season = dbutils.widgets.get('season')

In [0]:
## read raw data
if spark.catalog.tableExists("ipl_database.silver.potm"):
    df_raw = spark.table("ipl_database.bronze.potm_raw").filter(
        to_date(col("updated_at")) >= date.today()
    ) \
    .withColumn(
        "season",
        lit(season)
    )
else:
    df_raw = spark.table("ipl_database.bronze.potm_raw") \
    .withColumn(
        "season",
        lit(season)
    )
# df_raw.display()

## Clean raw data

In [0]:
df = (
  df_raw
  .dropDuplicates(['id'])
  .withColumn(
    "id",
    trim(col("id"))
  )
  .withColumn(
    "player",
    trim(col("player"))
  )
  .withColumn(
    "season",
    col("season").cast('int')
  )
  .select("id","player","season")
)


In [0]:
%skip

df.display()

## 01 Adding match UUID

In [0]:
match_uuid = spark.table("ipl_database.silver.match_details")

df_fin = (
  df.alias('a')
  .join(
    match_uuid.alias('b'),
    (col('a.id') == col('b.id')) & (col('a.season') == col('b.season')),
    'left'
  )
  .drop('id')
  .select('a.*','b.match_UUID')
)

In [0]:
%skip
df_fin.display()

In [0]:
%skip
if not spark.catalog.tableExists("ipl_database.silver.potm"):
  df_fin.write.format("delta").mode("overwrite").save(f"abfss://{container}@{storage_account}.dfs.core.windows.net/potm")
  spark.sql('''
            create table ipl_database.silver.potm
            using delta
            location 'abfss://silver@ipldatastorageaccount.dfs.core.windows.net/potm'
            ''')
  print("write complete!!!")
else:
  df_fin.write.format("delta").mode("append").save(f"abfss://{container}@{storage_account}.dfs.core.windows.net/potm")
  spark.sql('refresh table ipl_database.silver.potm')
  print("Append Sucess!!!")

In [0]:
%skip
%sql
describe history ipl_database.silver.potm

In [0]:
%skip
%sql
select * from ipl_database.silver.potm version as of 5

In [0]:
%skip
%sql
restore table ipl_database.silver.potm to version as of 5

In [0]:
%skip
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/potm")

history_df = delta_table.history()
history_df.display()

In [0]:

# delta_table.restoreToVersion(5)

In [0]:
# spark.sql('refresh table ipl_database.silver.potm')

In [0]:
%skip
%sql
select * from  ipl_database.silver.potm

In [0]:
%skip
%sql
drop table ipl_database.silver.potm

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.potm"):
        df_fin.createOrReplaceTempView("df_fin")
        spark.sql(f'''
                merge into ipl_database.silver.potm mc
                using df_fin nd
                on mc.player = nd.player and mc.season = nd.season
                when matched then update set *
                when not matched then insert *
                ''')
        # Upsert in not ideal here, switching to append process
        print("Table updated.")
    else:
        df_fin.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("ipl_database.silver.potm")
        print("Table created.")
except Exception as e:
    print(f"Error while write/append,{e}")